# ARBase × MPDD  드라이버 노트북

[OpenAnimals (ICCV 2025, Hou et al.)](https://arxiv.org/abs/2410.00204) 이 제안한 **ARBase**를 MPDD로 학습.
공식 코드/체크포인트가 아직 공개되지 않아, 논문 설명(IBN-ResNet50 + MGN 3-브랜치 + BoT)을 최대한 충실히 재구현한 버전입니다.

- 백본: ResNet50-IBN-a (ImageNet 사전학습, torch.hub `XingangPan/IBN-Net`), last stride 2→1
- 헤드: 전역/2-분할/3-분할 3개 브랜치 (전역급 특징 3개 + 로컬 스트라이프 5개 = 8-head)
- loss: triplet(전역급 3개) + label-smoothing CE(전체 8개), BNNeck
- 입력 384×384, 증강은 좌우 flip만, cosine LR (논문 6.2절)

모델/학습 코드는 [ML/scripts/arbase_model.py](../scripts/arbase_model.py), [ML/scripts/train_arbase.py](../scripts/train_arbase.py) 에 있고,
`datasets/`, `loss/` 는 `external/CLIP-ReID/` 안 CLIP-ReID 유틸을 `sys.path` 로 재사용합니다.

GPU 필요. 이 노트북은 `ML/notebooks/` 에서 실행한다고 가정합니다.

## 1. 학습

원본 MPDD(`dataset/raw/mpdd_release`)로 학습하면서, 5에폭마다 원본 MPDD query/gallery로 mAP를 찍어 best 체크포인트를 저장하고 5번 연속 개선 없으면 조기종료합니다.
결과는 `ML/checkpoints/arbase_mpdd_best.pth` 에 저장됩니다.

첫 실행 시 IBN-ResNet50 ImageNet 사전학습 가중치(~98MB)를 GitHub에서 받아옵니다.

In [ ]:
!python -u ../scripts/train_arbase.py \
  --root ../dataset/raw/mpdd_release/MPDD/pytorch \
  --epochs 60 \
  --batch_size 64 \
  --num_instances 4 \
  --img_size 384 \
  --eval_period 5 \
  --patience 5 \
  --num_workers 4 \
  --out ../checkpoints/arbase_mpdd_best.pth

## 2. 하드 eval — 다른 세 모델(CLIP-ReID/MegaDescriptor/PetFace)과 같은 기준으로 비교

케이스 단위(query 여러 장 평균 + gallery `--gallery_per_id` 장 캡) 채점. `MPDD_hard_corrupt` 는 열화된 query + YT-BB 방해꾼이 섞인 하드 eval 세트입니다.

In [ ]:
!python ../scripts/eval_case_level_wildlife.py \
  --model arbase \
  --root ../dataset/derived/MPDD_hard_corrupt/MPDD/pytorch \
  --weight ../checkpoints/arbase_mpdd_best.pth \
  --gallery_per_id 2 \
  --dump_errors ../dataset/derived/arbase_case_errors.csv